# Module 6 project: a transpiler comparison (browser version)

**Quantum Computing Intermediate · Module 6 · about 180 minutes** · notebook version 2026-10-05

**The question of this experiment.** Before a circuit runs on an IBM computer, the transpiler rewrites it in the computer's native gates, places it on physical qubits and adds swaps where qubits are not connected (Module 4). Its optimization level and its random seed change the result. **How much does the optimization level change the success of a circuit on a model of an IBM computer, how much does the transpiler's random seed change it, and which of these differences can 4,000 shots detect?** This is the **browser version**, for learners without a Google account. The transpiler and Qiskit Aer cannot run in a browser, so this version reads their results from a file saved with the same versions that the [Colab version](https://colab.research.google.com/github/CSU-Physics/quantum-computing-notebooks/blob/main/colab/level2/QC-L2-M6-project-transpiler-colab.ipynb) installs (qiskit 2.5.2, qiskit-aer 0.17.2, qiskit-ibm-runtime 0.50.0). You write the same functions and get the same quiz answers; the difference is that here you analyse the transpiler's results instead of running it.

In this project you run an experiment on a **noisy simulator**: a simulator that adds the errors of a real quantum computer, described by a **noise model**. You compare what you measure with what the noise model predicts, say how certain each result is, and say what the experiment cannot tell you. These are the skills of course outcome 5.

**The plan.**

1. Write three statistics functions (Step 1) and meet the computer (Step 2).
2. Write `mirror_circuit(bits)`: a circuit that should return its input, and see why it needs a barrier (Step 3).
3. Look at one run, with its uncertainty, and write `success_fraction(counts, bits)` (Step 4).
4. Compare 10 transpiler seeds at each optimization level and write `seed_spread(values)` (Step 5).
5. Test which optimization levels differ significantly (Step 6), and how success falls with the number of qubits (Step 7).
6. Get your verification value (Step 8), then look at the limits.

**How it is graded.** Your project quiz in Canvas (the one for this experiment, 20 points, pass at 16) asks for your verification value from the check cell, three results from this notebook, and four questions about uncertainty, interpretation and limits. Your project quiz gives you two numbers: BITS (a 4-bit string, 0 to 15) and SEED (100 to 999).

**Saving your work.** JupyterLite keeps your changes in this browser, so you can stop and come back. To keep a copy elsewhere, use **File → Download**. Run each code cell with **Shift + Enter**, in order; if you come back later, run the cells from the top again.

## Step 0: start Python and load the saved runs

Run the cell below. The first time, Python can take up to a minute to start in your browser, and Matplotlib takes a few more seconds to load.

In [ ]:
import json, math
import numpy as np
import matplotlib.pyplot as plt
import qsim
from qsim import QuantumCircuit, Statevector

saved = json.load(open("m6_transpiler_runs.json"))
print("Ready. qsim", qsim.__version__, "| saved runs made with", saved["versions"])

## Step 1: how certain is a result from shots?

A circuit that gives a result with probability p, run for S shots, gives that result k times. The number k is random: it follows the **binomial distribution**. The fraction f = k / S has

- mean p, and
- standard deviation **σ = √(p (1 − p) / S)**. This spread is called **shot noise**.

For example, with S = 4,000 shots and p = 0.933, σ = 0.0040: two runs of the same circuit typically differ by about this much. Four times as many shots halve σ.

**The course rule: 3σ.** A measured fraction f agrees with an expected value p when |f − p| ≤ 3σ. For a noisy simulator, p must be the **noise model's** expected value (computed exactly, without shots, in Step 2), not the ideal value from a circuit without noise. A correct run lands outside 3σ only about 3 times in 1,000 (0.27%).

**Comparing two results.** Two fractions f₁ and f₂, each from S independent shots, differ by f₂ − f₁, with standard deviation

**σ_diff = √( f₁(1 − f₁)/S + f₂(1 − f₂)/S )**.

Uncertainties of independent results add **in quadrature** (their squares add), so a difference is less certain than either result alone. A difference larger than 3σ_diff is **significant**: shot noise alone would almost never produce it. A smaller difference may be real, but this experiment cannot show it.

**Your task:** write the three functions below. They are used in every later step, and the check cell tests them.

- `shot_sigma(p, shots)`: σ = √(p (1 − p) / shots), the standard deviation of a fraction.
- `within_3_sigma(k, shots, p)`: `True` if the fraction k / shots is within 3 · shot_sigma(p, shots) of p (in either direction), otherwise `False`.
- `difference_sigma(f1, f2, shots)`: σ_diff for two fractions from `shots` shots each.

In [ ]:
def shot_sigma(p, shots):
    """The standard deviation of a fraction measured with `shots` shots, when the true probability is p."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete shot_sigma() first.")


def within_3_sigma(k, shots, p):
    """True if k successes out of `shots` agree with the probability p by the course's 3-sigma rule."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete within_3_sigma() first.")


def difference_sigma(f1, f2, shots):
    """The standard deviation of f2 - f1, for two fractions from `shots` independent shots each."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete difference_sigma() first.")

In [ ]:
print("shot_sigma(0.687, 4000) =", round(shot_sigma(0.687, 4000), 5), "  (expected 0.00733)")
print("within_3_sigma(2748, 4000, 0.687) =", within_3_sigma(2748, 4000, 0.687), "  (expected True: exactly 0.687)")
print("within_3_sigma(2655, 4000, 0.687) =", within_3_sigma(2655, 4000, 0.687), "  (expected False: 3.2 sigma too low)")
print("difference_sigma(0.687, 0.634, 4000) =", round(difference_sigma(0.687, 0.634, 4000), 5), "  (expected 0.01057)")

## Step 2: the computer and the saved runs

The computer is **FakePittsburgh**, a model of the IBM computer `ibm_pittsburgh` that comes with Qiskit (Module 4): 156 qubits on a heavy-hex lattice (each qubit connected to at most three others), native gates `cz`, `rz`, `sx` and `x`, and a noise model built from a snapshot of its calibration (median `cz` error 0.0015, median readout error 0.0045).

The saved runs hold, for 3, 4, 5 and 6 qubits, the four optimization levels 0 to 3 and the transpiler seeds 1 to 10: the number of two-qubit gates after transpiling, the physical qubits chosen, the counts of 4,000 shots on the noise model (with `seed_simulator` equal to the transpiler seed), and the noise model's **exact** probability of the right answer, `p_exact`, computed from the density matrix without shots. The prepared `saved_run(n, level, seed)` returns one of them.

In [ ]:
def saved_run(n, level, seed):
    """Prepared: the saved run for n qubits, an optimization level and a transpiler seed (1 to 10)."""
    return next(r for r in saved["runs"] if r["n"] == n and r["level"] == level and r["seed"] == seed)


r = saved_run(4, 3, 1)
print("4 qubits, level 3, seed 1: input", r["bits"], "|", r["two_qubit"], "two-qubit gates | depth", r["depth"],
      "| physical qubits", r["layout"], "| p_exact", r["p_exact"])

## Step 3: a circuit that should return its input

To compare transpiler settings, you need a circuit whose right answer is known for any size. A **mirror circuit** does a computation and then undoes it: here, prepare the input string, apply the QFT, then the inverse QFT. Without noise it returns the input every time, so any other result is an error.

**Your task:** write `mirror_circuit(bits)`:

1. X on every qubit q whose bit is 1 (qubit q is `bits[n - 1 - q]`, qubit 0 on the right);
2. the prepared `qft(qc, range(n))`;
3. **one barrier**, `qc.barrier()`;
4. the prepared `inverse_qft(qc, range(n))`;
5. measure qubit q into classical bit q.

The QFT and its inverse use `cp` and `swap` gates: for n qubits, n(n − 1)/2 + ⌊n/2⌋ two-qubit gates each, so 16 in all for 4 qubits.

In [ ]:
def qft(qc, qubits):
    """Prepared: the quantum Fourier transform on the given qubits (Qiskit's convention, with the final swaps)."""
    qubits = list(qubits)
    n = len(qubits)
    for j in reversed(range(n)):
        qc.h(qubits[j])
        for k in reversed(range(j)):
            qc.cp(np.pi / 2 ** (j - k), qubits[k], qubits[j])
    for j in range(n // 2):
        qc.swap(qubits[j], qubits[n - 1 - j])


def inverse_qft(qc, qubits):
    """Prepared: the inverse of qft(): the same gates in reverse order, with negative angles."""
    qubits = list(qubits)
    n = len(qubits)
    for j in range(n // 2):
        qc.swap(qubits[j], qubits[n - 1 - j])
    for j in range(n):
        for k in range(j):
            qc.cp(-np.pi / 2 ** (j - k), qubits[k], qubits[j])
        qc.h(qubits[j])


def mirror_circuit(bits):
    """Prepare |bits>, QFT, one barrier, inverse QFT, measure qubit q into classical bit q."""
    n = len(bits)
    qc = QuantumCircuit(n, n)
    # YOUR CODE HERE
    raise NotImplementedError("Complete mirror_circuit() first.")
    return qc

In [ ]:
qc = mirror_circuit("1011")
c = qc.copy()
c.remove_final_measurements()
print("without noise, mirror_circuit('1011') gives:", {k: round(float(v), 6) for k, v in Statevector(c).probabilities_dict().items() if v > 1e-9})
print("gates:", dict(qc.count_ops()))

**Why the barrier?** The transpiler's optimization passes look for gates that cancel, and a QFT followed by its inverse cancels completely. The table shows the saved 4-qubit circuit **without** the barrier, transpiled with seed 1: at level 3 every two-qubit gate is gone, only the three X gates remain, and the circuit tests nothing. A barrier tells the transpiler not to move or merge gates across it; it optimizes each half on its own.

In [ ]:
print("level   two-qubit gates without the barrier   with the barrier (seed 1)")
for nb in saved["no_barrier"]:
    print(f"  {nb['level']}                  {nb['two_qubit']:3d}                              {saved_run(4, nb['level'], 1)['two_qubit']:3d}")

## Step 4: one run and its uncertainty

**Your task:** write `success_fraction(counts, bits)`: the fraction of shots that returned `bits`. Use `counts.get(bits, 0)`, because a result that never appeared has no entry, and divide by the total number of shots in `counts`.

The cell takes the 4-qubit run at level 3 with seed 1 (input 1011), computes its success fraction and σ, and compares it with the noise model's exact value by the 3σ rule. The project quiz asks for this success fraction.

In [ ]:
def success_fraction(counts, bits):
    """The fraction of shots in counts that returned the string bits."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete success_fraction() first.")

In [ ]:
counts = saved_run(4, 3, 1)["counts"]
f = success_fraction(counts, "1011")
p = saved_run(4, 3, 1)["p_exact"]
print(f"success fraction {f:.4f}; noise model's exact value {p:.4f}")
print(f"sigma = {shot_sigma(p, 4000):.4f}; within 3 sigma: {within_3_sigma(counts.get('1011', 0), 4000, p)}")
top = sorted(counts.items(), key=lambda kv: -kv[1])[:5]
print("most frequent results:", top)

**What to notice.** About 93% of the shots return 1011. The wrong results are mostly one bit away from it: single errors, from a two-qubit gate or a readout. The difference from the exact value is a fraction of σ = 0.004: one run with 4,000 shots pins down this circuit's success to within about ±0.012 (3σ).

## Step 5: ten transpiler seeds

The transpiler makes random choices when it places qubits and adds swaps; `seed_transpiler` fixes them. Different seeds can give different physical qubits and different numbers of gates, so **the same circuit can succeed more or less often depending on the seed**. Is that variation larger than shot noise?

**Your task:** write `seed_spread(values)`: the **sample standard deviation** of a list of numbers, √(Σ(x − mean)² / (len − 1)). It measures how much results scatter from run to run. (`np.std(values, ddof=1)` does the same.)

The cell takes the 4-qubit runs at each level with seeds 1 to 10 and prints the mean success, the spread of the measured fractions, the spread of the exact values (the part due to the transpiler's choices alone), and the typical shot noise σ.

In [ ]:
def seed_spread(values):
    """The sample standard deviation of a list of numbers (divide by len(values) - 1)."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete seed_spread() first.")

In [ ]:
print("level   two-qubit gates   mean success   spread (measured)   spread (exact)   shot sigma")
summary = {}
for lvl in range(4):
    runs = [saved_run(4, lvl, s) for s in range(1, 11)]
    fr = [success_fraction(r["counts"], "1011") for r in runs]
    ex = [r["p_exact"] for r in runs]
    tq = [r["two_qubit"] for r in runs]
    summary[lvl] = (np.mean(fr), seed_spread(fr))
    print(f"  {lvl}       {min(tq):3d} to {max(tq):3d}         {np.mean(fr):.4f}         {seed_spread(fr):.4f}            {seed_spread(ex):.4f}         {shot_sigma(np.mean(fr), 4000):.4f}")

**What to notice.**

- At **level 1**, every seed gives the same 54 two-qubit gates and the same exact value, so the measured spread (0.004) is just shot noise.
- At **level 0** the spread of the measured fractions (0.011) is clearly larger than shot noise (0.007), and the exact values alone spread by 0.010: the seeds give different routings (60 to 66 two-qubit gates) on different qubits.
- At levels 2 and 3 the seeds give 32 to 38 two-qubit gates, and the transpiler's choices add a spread about as large as shot noise.

**The lesson for uncertainty:** σ from one run says how well that run measured **its** circuit. If the question is about a transpiler setting, not one transpiled circuit, the uncertainty must include the variation between seeds, and one run is not enough to measure it.

## Step 6: which optimization levels differ significantly?

To compare two levels, use the **mean over the 10 seeds**. Its uncertainty is the **standard error of the mean**, SEM = seed_spread / √10, which includes both shot noise and the seed-to-seed variation. For two means, σ_diff = √(SEM₁² + SEM₂²), the same rule as `difference_sigma`. The cell compares levels 1 and 3, and 2 and 3, for 4, 5 and 6 qubits.

In [ ]:
def level_mean(n, level):
    bits = saved["widths"][str(n)]
    fr = [success_fraction(saved_run(n, level, s)["counts"], bits) for s in range(1, 11)]
    return np.mean(fr), seed_spread(fr) / math.sqrt(len(fr))


for n in (4, 5, 6):
    for a, b in ((1, 3), (2, 3)):
        (ma, sa), (mb, sb) = level_mean(n, a), level_mean(n, b)
        d, sd = mb - ma, math.sqrt(sa ** 2 + sb ** 2)
        print(f"{n} qubits: level {a} {ma:.4f} ± {sa:.4f}, level {b} {mb:.4f} ± {sb:.4f}: difference {d:+.4f} = {d / sd:+.1f} sigma_diff,",
              "significant" if abs(d) > 3 * sd else "not significant")

**What to notice.** Level 3 beats level 1 by 3 to 5 percentage points at every size, many σ_diff: a clear result. Level 3 against level 2 is subtler: significantly better for 4 and 5 qubits (by 0.010 and 0.023, with the same gate counts: level 3 chooses physical qubits with lower errors), but for 6 qubits the two levels give the same success within 0.001, far below σ_diff. "Level 3 is better" is true for some circuits and not shown for others.

## Step 7: success against the number of qubits

The cell shows the mean success at level 3 for 3 to 6 qubits, with its SEM, and the mean number of two-qubit gates after transpiling. The project quiz asks for the mean success with 6 qubits.

If every two-qubit gate failed with the same probability ε and nothing else went wrong, the success would be about (1 − ε)^N₂ for N₂ two-qubit gates. The last column inverts this, ε ≈ 1 − success^(1/N₂): an **effective error per two-qubit gate**, which also absorbs the readout and one-qubit errors.

In [ ]:
print("qubits   two-qubit gates (mean)   mean success ± SEM    effective error per two-qubit gate")
ns, means, sems = [3, 4, 5, 6], [], []
for n in ns:
    m, se = level_mean(n, 3)
    n2 = np.mean([saved_run(n, 3, s)["two_qubit"] for s in range(1, 11)])
    means.append(m); sems.append(se)
    print(f"  {n}            {n2:5.1f}                {m:.4f} ± {se:.4f}            {1 - m ** (1 / n2):.4f}")

plt.figure(figsize=(6.5, 3.3))
plt.errorbar(ns, means, yerr=[3 * s for s in sems], fmt="C0o-", capsize=4, label="level 3, mean of 10 seeds, ±3 SEM")
plt.xticks(ns); plt.xlabel("qubits"); plt.ylabel("success fraction"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

**What to notice.** The success falls from 0.968 with 3 qubits to 0.861 with 6, while the two-qubit gates grow from about 17 to 92: the QFT has a `cp` between every pair of qubits, so its gate count grows as n², and on a heavy-hex chip most pairs need swaps. The effective error per two-qubit gate is about 0.002, close to the computer's median `cz` error (0.0015) plus a share of the readout errors.

## Step 8: your verification value

Open your **project quiz for this experiment** in Canvas. Question 1 shows your BITS (0 to 15: a 4-bit input string, for example 11 is 1011) and SEED (100 to 999). Type them below and run the next cell. The check cell tests your three statistics functions, `mirror_circuit()`, `success_fraction()` and `seed_spread()`. Only if everything passes does it print your **verification value**: in the course's saved runs, the number of shots out of 4,000 in which the mirror circuit for BITS, transpiled at level 3 with `seed_transpiler = SEED` and run on the noise model with `seed_simulator = SEED`, returned BITS.

In [ ]:
BITS = -1     # your number from Canvas, for example 11 (the string 1011)
SEED = 0      # your seed from Canvas, for example 512

In [ ]:
# The course's check code. You do not need to read it; it is here so that you can.
import math

import numpy as np

try:                                    # real Qiskit, if it is installed
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
except ImportError:                     # the browser: the course simulator
    from qsim import QuantumCircuit, Statevector

ONE_QUBIT_GATES = ["h", "x", "rx", "ry", "rz", "p", "t", "tdg", "s", "sdg", "sx"]
TWO_QUBIT_GATES = ["cx", "cz", "cp", "rzz", "swap"]
SHOTS = 4000

# ---------------------------------------------------------------- an exact noisy simulator (NumPy only)
_S2 = 1 / math.sqrt(2)
_FIXED = {
    "h": np.array([[_S2, _S2], [_S2, -_S2]], dtype=complex),
    "x": np.array([[0, 1], [1, 0]], dtype=complex),
    "t": np.diag([1, np.exp(1j * math.pi / 4)]),
    "tdg": np.diag([1, np.exp(-1j * math.pi / 4)]),
    "s": np.diag([1, 1j]),
    "sdg": np.diag([1, -1j]),
}


def _matrix(name, params):
    """The gate's matrix in Qiskit's convention (for two-qubit gates, the first qubit is the right-hand factor)."""
    if name in _FIXED:
        return _FIXED[name]
    if name == "rx":
        c, s = math.cos(params[0] / 2), math.sin(params[0] / 2)
        return np.array([[c, -1j * s], [-1j * s, c]])
    if name == "rz":
        return np.diag([np.exp(-0.5j * params[0]), np.exp(0.5j * params[0])])
    if name == "p":
        return np.diag([1, np.exp(1j * params[0])])
    if name == "cx":                     # control = first qubit (the low bit of the 4 x 4 index)
        return np.array([[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0]], dtype=complex)
    if name == "cz":
        return np.diag([1, 1, 1, -1]).astype(complex)
    if name == "cp":
        return np.diag([1, 1, 1, np.exp(1j * params[0])])
    if name == "rzz":
        a = np.exp(-0.5j * params[0])
        b = np.exp(0.5j * params[0])
        return np.diag([a, b, b, a])
    if name == "swap":
        return np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1]], dtype=complex)
    raise ValueError(f"unknown gate {name}")


def _apply(rho, u, qubits, n):
    """rho -> U rho U^dagger, with U acting on the given qubits (qubits[0] is the low bit of U's index)."""
    k = len(qubits)
    t = rho.reshape([2] * (2 * n))
    # axis of qubit q in the row index is n - 1 - q; in the column index 2n - 1 - q
    ut = u.reshape([2] * (2 * k))           # axes: out bits (high to low), in bits (high to low)
    row_axes = [n - 1 - q for q in reversed(qubits)]
    col_axes = [2 * n - 1 - q for q in reversed(qubits)]
    t = np.tensordot(ut, t, axes=(list(range(k, 2 * k)), row_axes))
    t = np.moveaxis(t, list(range(k)), row_axes)
    t = np.tensordot(t, ut.conj(), axes=(col_axes, list(range(k, 2 * k))))
    t = np.moveaxis(t, list(range(2 * n - k, 2 * n)), col_axes)
    return t.reshape(2 ** n, 2 ** n)


def _depolarize(rho, lam, qubits, n):
    """rho -> (1 - lam) rho + lam (I / 2^k on the given qubits) x (rho traced over them)."""
    if lam == 0:
        return rho
    k = len(qubits)
    t = rho.reshape([2] * (2 * n))
    row_axes = [n - 1 - q for q in qubits]
    col_axes = [2 * n - 1 - q for q in qubits]
    letters = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    sub = list(letters[:2 * n])
    for ra, ca in zip(row_axes, col_axes):
        sub[ca] = sub[ra]
    keep = [i for i in range(2 * n) if i not in row_axes + col_axes]
    red = np.einsum("".join(sub) + "->" + "".join(sub[i] for i in keep), t)
    eye = np.eye(2 ** k).reshape([2] * (2 * k))      # identity: row bit i pairs with column bit i
    full = np.multiply.outer(red, eye)
    full = np.moveaxis(full, list(range(len(keep), 2 * n)), row_axes[::-1] + col_axes[::-1])
    return (1 - lam) * rho + lam * full.reshape(2 ** n, 2 ** n) / 2 ** k


def exact_probabilities(ops, n, measured, p2, readout):
    """Exact probabilities of the recorded results under the course noise model.

    ops: list of (name, qubits, params) applied to |0...0>; measured: the qubits measured at the end, measured[j] into
    classical bit j. Returns an array over the 2^len(measured) recorded results (bit j of the index is classical bit j).
    """
    rho = np.zeros((2 ** n, 2 ** n), dtype=complex)
    rho[0, 0] = 1
    for name, qubits, params in ops:
        rho = _apply(rho, _matrix(name, params), list(qubits), n)
        if name in TWO_QUBIT_GATES:
            rho = _depolarize(rho, p2, list(qubits), n)
        elif name in ONE_QUBIT_GATES:
            rho = _depolarize(rho, p2 / 10, list(qubits), n)
    probs = np.real(np.diag(rho)).clip(min=0)
    probs = probs / probs.sum()
    m = len(measured)
    idx = np.arange(2 ** n)
    code = np.zeros(2 ** n, dtype=int)
    for j, q in enumerate(measured):
        code |= ((idx >> q) & 1) << j
    dist = np.zeros(2 ** m)
    np.add.at(dist, code, probs)
    conf = np.array([[1 - readout, readout], [2 * readout, 1 - 2 * readout]])
    t = dist.reshape([2] * m)
    for j in range(m):
        ax = m - 1 - j
        t = np.moveaxis(np.tensordot(t, conf, axes=([ax], [0])), -1, ax)
    dist = t.reshape(-1).clip(min=0)
    return dist / dist.sum()


def build_circuit(ops, n, measured):
    """The ops as a QuantumCircuit (Qiskit or qsim), with the measurements at the end."""
    qc = QuantumCircuit(n, len(measured)) if measured else QuantumCircuit(n)
    for name, qubits, params in ops:
        getattr(qc, name)(*params, *qubits)
    for j, q in enumerate(measured):
        qc.measure(q, j)
    return qc


# ---------------------------------------------------------------- helpers for the checks
def _op(inst):
    return getattr(inst, "operation", inst)


def _name(inst):
    return getattr(_op(inst), "name", "")


def _qubit_indices(qc, inst):
    qs = getattr(inst, "qubits", ())
    try:
        return [qc.find_bit(q).index for q in qs]
    except Exception:  # noqa: BLE001
        return [int(q) for q in qs]


def _clbit_indices(qc, inst):
    cs = getattr(inst, "clbits", ())
    try:
        return [qc.find_bit(c).index for c in cs]
    except Exception:  # noqa: BLE001
        return [int(c) for c in cs]


def measurement_map(qc):
    """{qubit: classical bit} for the measurements in a circuit."""
    return {_qubit_indices(qc, i)[0]: _clbit_indices(qc, i)[0] for i in qc.data if _name(i) == "measure"}


def big_gates(qc):
    """Names of gates on 3 or more qubits (the course noise model puts no error on them)."""
    return sorted({_name(i) for i in qc.data if _name(i) not in ("barrier", "measure") and len(_qubit_indices(qc, i)) >= 3})


def two_qubit_count(qc):
    """The number of two-qubit gates in a circuit (barriers not counted)."""
    return sum(1 for i in qc.data if _name(i) not in ("barrier", "measure") and len(_qubit_indices(qc, i)) == 2)


def ops_two_qubit_count(ops):
    return sum(1 for name, qubits, params in ops if len(qubits) == 2)


def ideal_probabilities(qc):
    """Probabilities of all qubits' values (index bit q = qubit q) for a circuit without its final measurements."""
    c = qc.copy()
    c.remove_final_measurements()
    return np.abs(np.asarray(Statevector(c).data, dtype=complex)) ** 2


def counts_total(counts):
    try:
        return int(sum(int(v) for v in counts.values()))
    except Exception:  # noqa: BLE001
        return -1


# ---------------------------------------------------------------- the statistics functions
def _close(a, b, tol=1e-9):
    try:
        return abs(float(a) - float(b)) <= tol * max(1.0, abs(float(b)))
    except (TypeError, ValueError):
        return False


def test_statistics(shot_sigma, within_3_sigma, difference_sigma):
    """Returns (passed, messages)."""
    msgs = []
    cases = [(0.5, 100), (0.687, 4000), (0.1, 1000), (0.0, 50), (1.0, 50), (0.9206, 4000)]
    for p, n in cases:
        want = math.sqrt(p * (1 - p) / n)
        try:
            got = shot_sigma(p, n)
        except NotImplementedError:
            return False, ["shot_sigma() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"shot_sigma({p}, {n}) raised {type(e).__name__}: {e}"]
        if not _close(got, want):
            hint = ""
            if _close(got, math.sqrt(p * (1 - p) * n)):
                hint = " That is the uncertainty of the count k; divide by shots for the fraction: sqrt(p * (1 - p) / shots)."
            elif _close(got, p * (1 - p) / n):
                hint = " Take the square root: sqrt(p * (1 - p) / shots)."
            elif _close(got, math.sqrt(p / n)) or _close(got, math.sqrt(1 / n)):
                hint = " Use p * (1 - p), the binomial variance of one shot."
            return False, [f"shot_sigma({p}, {n}) returned {got}; expected {want:.6g}.{hint}"]
    msgs.append("shot_sigma(): passed (6 cases).")

    w_cases = [(2748, 4000, 0.687, True), (2655, 4000, 0.687, False), (2841, 4000, 0.687, False),
               (2700, 4000, 0.687, True), (2821, 4000, 0.687, True), (55, 100, 0.5, True), (66, 100, 0.5, False),
               (34, 100, 0.5, False), (3700, 4000, 0.9206, True), (3620, 4000, 0.9206, False)]
    for k, n, p, want in w_cases:
        try:
            got = within_3_sigma(k, n, p)
        except NotImplementedError:
            return False, msgs + ["within_3_sigma() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, msgs + [f"within_3_sigma({k}, {n}, {p}) raised {type(e).__name__}: {e}"]
        if bool(got) != want or not isinstance(got, (bool, np.bool_)):
            d = abs(k / n - p) / max(math.sqrt(p * (1 - p) / n), 1e-300)
            hint = ""
            if not isinstance(got, (bool, np.bool_)):
                hint = " Return True or False."
            elif k > n * p and not want:
                hint = " Check both directions: a result can be too high as well as too low (use abs())."
            elif 2 < d <= 3:
                hint = " The course rule allows up to 3 standard deviations."
            else:
                hint = " Compare the fraction k / shots with p, using shot_sigma(p, shots)."
            return False, msgs + [f"within_3_sigma({k}, {n}, {p}) returned {got}; expected {want} "
                                  f"(the result is {d:.2f} standard deviations from p).{hint}"]
    msgs.append("within_3_sigma(): passed (10 cases).")

    d_cases = [(0.687, 0.634, 4000), (0.5, 0.5, 100), (0.9, 0.1, 1000), (0.2, 0.25, 4000)]
    for f1, f2, n in d_cases:
        want = math.sqrt(f1 * (1 - f1) / n + f2 * (1 - f2) / n)
        try:
            got = difference_sigma(f1, f2, n)
        except NotImplementedError:
            return False, msgs + ["difference_sigma() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, msgs + [f"difference_sigma({f1}, {f2}, {n}) raised {type(e).__name__}: {e}"]
        if not _close(got, want):
            s1, s2 = math.sqrt(f1 * (1 - f1) / n), math.sqrt(f2 * (1 - f2) / n)
            hint = ""
            if _close(got, s1 + s2):
                hint = " Uncertainties add in quadrature: sqrt(sigma1**2 + sigma2**2), not sigma1 + sigma2."
            elif _close(got, abs(s1 - s2)):
                hint = " The uncertainty of a difference is larger than either one: sqrt(sigma1**2 + sigma2**2)."
            elif _close(got, want ** 2):
                hint = " Take the square root."
            return False, msgs + [f"difference_sigma({f1}, {f2}, {n}) returned {got}; expected {want:.6g}.{hint}"]
    msgs.append("difference_sigma(): passed (4 cases).")
    return True, msgs


def personal_noise_ok(P2, READOUT):
    """Checks the personal noise numbers from Canvas; returns (ok, message, p2, readout)."""
    try:
        p2, ro = float(P2), float(READOUT)
    except (TypeError, ValueError):
        return False, "P2 and READOUT must be the numbers shown in your Canvas project quiz.", None, None
    if not (0.005 - 1e-12 <= p2 <= 0.030 + 1e-12) or abs(round(p2, 3) - p2) > 1e-9:
        return False, "P2 is a number from 0.005 to 0.030 with three decimals. Copy it again from the project quiz.", None, None
    if not (0.005 - 1e-12 <= ro <= 0.030 + 1e-12) or abs(round(ro, 3) - ro) > 1e-9:
        return False, "READOUT is a number from 0.005 to 0.030 with three decimals. Copy it again from the project quiz.", None, None
    return True, "", round(p2, 3), round(ro, 3)


MIRROR_CASES = ["1", "10", "101", "0110", "10011"]


def reference_qft_ops(qubits):
    qubits = list(qubits)
    n = len(qubits)
    ops = []
    for j in reversed(range(n)):
        ops.append(("h", [qubits[j]], []))
        for k in reversed(range(j)):
            ops.append(("cp", [qubits[k], qubits[j]], [math.pi / 2 ** (j - k)]))
    ops += [("swap", [qubits[j], qubits[n - 1 - j]], []) for j in range(n // 2)]
    return ops


def reference_prefix_ops(bits):
    n = len(bits)
    return [("x", [q], []) for q in range(n) if bits[n - 1 - q] == "1"] + reference_qft_ops(range(n))


def _prefix(qc, stop):
    c = qc.copy_empty_like()
    for inst in list(qc.data)[:stop]:
        c.append(inst)
    return c


def _state(qc):
    return np.asarray(Statevector(qc).data, dtype=complex)


def _test_mirror(mirror_circuit):
    for bits in MIRROR_CASES:
        n = len(bits)
        label = f"mirror_circuit('{bits}')"
        try:
            qc = mirror_circuit(bits)
        except NotImplementedError:
            return False, ["mirror_circuit() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"{label} raised {type(e).__name__}: {e}"]
        if not hasattr(qc, "data") or getattr(qc, "num_qubits", None) != n:
            return False, [f"{label} should return a QuantumCircuit on {n} qubits."]
        mm = measurement_map(qc)
        if mm != {q: q for q in range(n)}:
            return False, [f"{label} should end by measuring qubit q into classical bit q, for every qubit (found {mm or 'no measurements'})."]
        names = [_name(i) for i in qc.data]
        barriers = [k for k, nm in enumerate(names) if nm == "barrier"]
        if len(barriers) != 1:
            return False, [f"{label} has {len(barriers)} barriers; put exactly one, between qft and inverse_qft. Without it, "
                           "the transpiler at levels 2 and 3 can cancel the QFT against its inverse and nothing is tested; "
                           "extra barriers stop it from optimizing inside the QFT."]
        try:
            before = _state(_prefix(qc, barriers[0]))
        except Exception as e:  # noqa: BLE001
            return False, [f"The part of {label} before the barrier cannot be simulated: {type(e).__name__}: {e}"]
        want = _state(build_circuit(reference_prefix_ops(bits), n, []))
        if abs(np.vdot(want, before)) ** 2 < 1 - 1e-9:
            hint = " Before the barrier: X on each qubit whose bit is 1 (qubit 0 is the rightmost bit), then qft(qc, range(n))."
            rev = _state(build_circuit(reference_prefix_ops(bits[::-1]), n, []))
            if abs(np.vdot(rev, before)) ** 2 > 1 - 1e-9 and bits != bits[::-1]:
                hint = " The X gates are on the reversed string: qubit q gets an X when bits[n - 1 - q] is '1'."
            return False, [f"The state of {label} at the barrier differs from the expected one.{hint}"]
        probs = ideal_probabilities(qc)
        if probs[int(bits, 2)] < 1 - 1e-9:
            return False, [f"Without noise, {label} returns '{bits}' with probability {probs[int(bits, 2)]:.4f}; it should be 1. "
                           "After the barrier, apply the prepared inverse_qft on the same qubits."]
        want2 = 2 * ops_two_qubit_count(reference_qft_ops(range(n)))
        if two_qubit_count(qc) != want2:
            return False, [f"{label} has {two_qubit_count(qc)} two-qubit gates; the project's circuit has {want2} "
                           "(the prepared qft and inverse_qft and nothing else). Every extra gate changes what the transpiler gets."]
    return True, [f"mirror_circuit(): passed ({len(MIRROR_CASES)} cases, 1 to 5 qubits)."]


SUCCESS_CASES = [({"1011": 3600, "1010": 300, "0011": 100}, "1011"), ({"01": 7, "10": 3}, "10"),
                 ({"000": 5, "111": 5}, "101"), ({"10110": 1}, "10110")]


def _test_success(success_fraction):
    for counts, bits in SUCCESS_CASES:
        want = counts.get(bits, 0) / sum(counts.values())
        try:
            got = success_fraction(dict(counts), bits)
        except NotImplementedError:
            return False, ["success_fraction() is not written yet."]
        except KeyError:
            return False, [f"success_fraction({counts}, '{bits}') raised KeyError: use counts.get(bits, 0), because a "
                           "result that never appeared has no entry."]
        except Exception as e:  # noqa: BLE001
            return False, [f"success_fraction({counts}, '{bits}') raised {type(e).__name__}: {e}"]
        try:
            gotf = float(got)
        except (TypeError, ValueError):
            return False, [f"success_fraction(counts, '{bits}') should return a number; it returned {got!r}."]
        if abs(gotf - want) > 1e-12:
            hint = ""
            if abs(gotf - counts.get(bits, 0)) < 1e-12:
                hint = " Divide by the total number of shots, sum(counts.values())."
            elif abs(gotf - counts.get(bits, 0) / 4000) < 1e-12 and sum(counts.values()) != 4000:
                hint = " Divide by the total of this counts dictionary, not by 4000."
            return False, [f"success_fraction({counts}, '{bits}') returned {gotf}; expected {want}.{hint}"]
    return True, [f"success_fraction(): passed ({len(SUCCESS_CASES)} cases)."]


SPREAD_CASES = [[0.9, 0.92, 0.94], [0.8589, 0.8, 0.88, 0.85, 0.83], [1, 1, 1, 1], [0.5, 0.7]]


def _test_spread(seed_spread):
    for vals in SPREAD_CASES:
        want = float(np.std(vals, ddof=1))
        try:
            got = seed_spread(list(vals))
        except NotImplementedError:
            return False, ["seed_spread() is not written yet."]
        except Exception as e:  # noqa: BLE001
            return False, [f"seed_spread({vals}) raised {type(e).__name__}: {e}"]
        try:
            gotf = float(got)
        except (TypeError, ValueError):
            return False, [f"seed_spread({vals}) should return a number; it returned {got!r}."]
        if abs(gotf - want) > 1e-12:
            hint = ""
            if abs(gotf - float(np.std(vals))) < 1e-12:
                hint = " Divide by len(values) - 1 (the sample standard deviation): np.std(values, ddof=1)."
            elif abs(gotf - float(np.var(vals, ddof=1))) < 1e-12:
                hint = " Take the square root of the variance."
            elif abs(gotf - want / math.sqrt(len(vals))) < 1e-12:
                hint = " That is the uncertainty of the mean; seed_spread() is the spread of the values themselves."
            return False, [f"seed_spread({vals}) returned {gotf}; expected {want:.6f}.{hint}"]
    return True, [f"seed_spread(): passed ({len(SPREAD_CASES)} cases)."]


def personal_value(bits, seed, saved):
    for d in saved.get("personal", []):
        if int(d["bits"]) == int(bits) and int(d["seed"]) == int(seed):
            return int(d["success"])
    return None


def check_l2_m6_transpiler(mirror_circuit, success_fraction, seed_spread, shot_sigma, within_3_sigma, difference_sigma,
                           BITS, SEED, saved):
    try:
        b, s = int(BITS), int(SEED)
        if b != float(BITS) or s != float(SEED) or not 0 <= b <= 15 or not 100 <= s <= 999:
            raise ValueError
    except (TypeError, ValueError):
        return False, ["BITS is a whole number from 0 to 15 and SEED from 100 to 999. Copy them again from your project quiz."], None
    if not isinstance(saved, dict) or "personal" not in saved:
        return False, ["The saved runs are not loaded: run the cell of Step 0 that loads m6_transpiler_runs.json."], None
    msgs, passed = [], True
    for good, mm in (test_statistics(shot_sigma, within_3_sigma, difference_sigma), _test_mirror(mirror_circuit),
                     _test_success(success_fraction), _test_spread(seed_spread)):
        msgs += mm
        passed = passed and good
        if not good:
            break
    if not passed:
        return False, msgs, None
    value = personal_value(b, s, saved)
    if value is None:
        return False, msgs + [f"BITS = {b} with SEED = {s} is not one of the course's saved runs. Copy both numbers again "
                              "from your project quiz."], None
    return True, msgs, value


passed, messages, value = check_l2_m6_transpiler(mirror_circuit, success_fraction, seed_spread, shot_sigma,
                                                 within_3_sigma, difference_sigma, BITS, SEED, saved)
for m in messages:
    print(m)
if passed:
    print("\nAll tests passed. Your verification value is", value)
else:
    print("\nNot yet: fix the item above and run this cell again.")

## Limits of this experiment

- **One model of one computer.** FakePittsburgh is a snapshot of one calibration. The real `ibm_pittsburgh` is recalibrated daily, its errors drift, and the best qubits today may not be the best tomorrow; a level-3 advantage that comes from choosing good qubits depends on that snapshot.
- **What the noise model leaves out.** Gate and readout errors and decoherence are in it; crosstalk between neighbouring qubits, leakage out of the qubit states, and errors that repeat the same way every time are not. On the real computer, results are usually somewhat worse than on its noise model.
- **One kind of circuit.** A mirror circuit returns its input, which makes success easy to define. Circuits whose answer is a distribution (QAOA, phase estimation of 1/3) need other measures, and transpiler settings can rank differently for them.
- **Ten seeds.** The seed-to-seed spread is itself estimated from 10 values, so it is uncertain too (by roughly a quarter of its size).

These limits are not failures of your experiment: every experiment has them. Saying what a result does and does not show is part of reporting it.

## Optional: run it on a real IBM quantum computer

If you have an IBM Quantum account with an Open Plan instance (see **Set Up Your Tools** in Start Here), the [Module 6 hardware notebook in Colab](https://colab.research.google.com/github/CSU-Physics/quantum-computing-notebooks/blob/main/colab/level2/QC-L2-M6-hardware-colab.ipynb) runs the project's reference circuit on a real computer and compares the result with the ideal value, with this course's noise model, and with IBM's own noise model of that computer. Your API key stays in Colab's **Secrets**; never type it into a cell. The Open Plan gives up to 10 minutes of quantum computer time every 28 days, and one run of 4,000 shots uses a few seconds of it, but the queue can take from seconds to hours. This step is not graded, and the badge does not depend on it.

## Optional: a short summary of your experiment

Not graded. Writing five short answers is a good way to prepare for the project quiz, and it is how results are reported in a lab notebook or a paper. Double-click this cell to edit it.

1. **Question.** What did you want to find out?
2. **Method.** Which circuit, which noise model (P2, READOUT), how many shots?
3. **Result.** Your main numbers, each with its uncertainty (for example 0.687 ± 0.007, with σ from Step 1).
4. **Interpretation.** What do the numbers say about the question? Which differences are significant (more than 3σ)?
5. **Limits.** What does this experiment not show?

**Next in Canvas:** the Module 6 project quiz for this experiment, the transpiler comparison, and the Module 6 time log.